In [2]:
%load_ext autoreload
%autoreload 2
# Init everything here
#!python
# -*- coding: utf-8 -*-

import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
from src.lib import boot, ui, vision, logger
from src.lib import rok_data

FARM_INSTANCE = "farm1"
MAIN_INSTANCE = "main"
# instance = FARM_INSTANCE
instance = FARM_INSTANCE

logger.setup_logger()
boot.init_instance(instance, rok_ready=False)

[INFO] 2025-10-13 17:45:56 - 
-------------------- INIT START --------------------
[2025-10-13 17:45:56] Instance name: 'farm1'
[INFO] 2025-10-13 17:45:56 - Refreshing ADB server...
[INFO] 2025-10-13 17:45:56 - Normal mode: restarting ADB server


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[INFO] 2025-10-13 17:46:01 - ADB server refreshed.
[WARNING] 2025-10-13 17:46:01 - Failed to init LDInstance: farm1
[DEBUG] 2025-10-13 17:46:01 - Start instance 'farm1', wait for 15s
[DEBUG] 2025-10-13 17:46:03 - Instance 'farm1' started
[DEBUG] 2025-10-13 17:46:03 - Verify adb connection
[DEBUG] 2025-10-13 17:46:03 - Init ADBInstance with LDPInstance farm1
[DEBUG] 2025-10-13 17:46:03 - _pre_running_devices: set()
[DEBUG] 2025-10-13 17:46:03 - Fresh ADB initialization. Wait for device up
[WARNING] 2025-10-13 17:46:03 - Device not ready, wait for 5s ... set()
[WARNING] 2025-10-13 17:46:08 - Device not ready, wait for 5s ... set()
[WARNING] 2025-10-13 17:46:13 - Device not ready, wait for 5s ... set()
[DEBUG] 2025-10-13 17:46:18 - Device ready: {'emulator-5554'}
[INFO] 2025-10-13 17:46:18 - Desired device to connect emulator-5554
[INFO] 2025-10-13 17:46:18 - Connected adb emulator-5554 with instance farm1
[INFO] 2025-10-13 17:46:19 - Init Rise of Kingdoms
[INFO] 2025-10-13 17:46:19 - Wai

In [2]:
from pymongo import MongoClient

# Default connection URI (MongoDB runs on port 27017)
client = MongoClient("mongodb://localhost:27017/")

# List all databases
print("Databases:", client.list_database_names())

Databases: ['admin', 'config', 'local']


In [17]:
db = client["rok_prod"]
characters = db["characters"]
accounts = db["accounts"]
rss_orders = db["rss_orders"]
print(characters)

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'rok_prod'), 'characters')


In [5]:
import yaml

with open("../profile.yml", "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)
print(data["characters"])

{'main': {'name': 'Dei', 'slot_number': 1}, '1f1': {'name': 'Dei 1F1', 'ch': 24, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 2}, '1f2': {'name': 'Dei 1F2', 'ch': 24, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 3}, '1f3': {'name': 'Dei 1F3', 'ch': 22, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 4}, '1f4': {'name': 'Dei 1F4', 'ch': 22, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 5}, '2f1': {'name': 'Dei 2F1', 'ch': 24, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 1}, '2f2': {'name': 'Dei 2F2', 'ch': 24, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 2}, '2f3': {'name': 'Dei 2F3', 'ch': 23, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 3}, '2f4': {'name': 'Dei 2F4', 'ch': 22, 'rss_order': 'V2', 'rss_level': 8, 'slot_number': 4}}


In [ ]:
from pprint import pprint as pp

docs = []
for key, value in data["characters"].items():
    value["_id"] = key  # use YAML key as unique ID (e.g., "1f1", "main")
    docs.append(value)
pp(docs)

In [ ]:
acc = []
for key, value in data["accounts"].items():
    value["_id"] = key  # use YAML key as unique ID (e.g., "1f1", "main")
    acc.append(value)
pp(acc)

if acc:
    result = accounts.insert_many(acc)
    print(f"✅ Inserted {len(result.inserted_ids)} accounts into MongoDB.")
else:
    print("⚠️ No accounts to insert.")

[{'_id': '22898455',
  'characters': ['1f1', '1f2', '1f3', '1f4'],
  'name': 'main_account'},
 {'_id': '24611449',
  'characters': ['2f1', '2f2', '2f3', '2f4'],
  'name': 'sub_account_1'}]


In [ ]:
orders = []
for key, value in data["rss_orders"].items():
    orders.append({
        "_id": key,          # use V1, V2, V3... as ID
        "resources": value   # list of resources
    })
pp(orders)

if orders:
    result = rss_orders.insert_many(orders)
    print(f"✅ Inserted {len(result.inserted_ids)} rss_orders into MongoDB.")
else:
    print("⚠️ No rss_orders to insert.")

[{'_id': 'V1', 'resources': ['gold', 'gold', 'stone', 'wood', 'food']},
 {'_id': 'V2', 'resources': ['gold', 'gold', 'stone', 'food', 'food']},
 {'_id': 'V3', 'resources': ['gold', 'gold', 'gold', 'gold', 'gold']},
 {'_id': 'V4', 'resources': ['gold', 'gold', 'gold', 'stone', 'food']},
 {'_id': 'V5', 'resources': ['gold', 'stone', 'food', 'food', 'food']}]
✅ Inserted 5 rss_orders into MongoDB.


In [3]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))


from src.lib.rok_data import CharactersDB, AccountsDB

CharactersDB().get_all()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[Character(_id='main', name='Dei', slot_number=1, ch=25, rss_order='V1', rss_level=8),
 Character(_id='1f1', name='Dei 1F1', slot_number=2, ch=24, rss_order='V2', rss_level=8),
 Character(_id='1f2', name='Dei 1F2', slot_number=3, ch=24, rss_order='V2', rss_level=8),
 Character(_id='1f3', name='Dei 1F3', slot_number=4, ch=22, rss_order='V2', rss_level=8),
 Character(_id='1f4', name='Dei 1F4', slot_number=5, ch=22, rss_order='V2', rss_level=8),
 Character(_id='2f1', name='Dei 2F1', slot_number=1, ch=24, rss_order='V2', rss_level=8),
 Character(_id='2f2', name='Dei 2F2', slot_number=2, ch=24, rss_order='V2', rss_level=8),
 Character(_id='2f3', name='Dei 2F3', slot_number=3, ch=23, rss_order='V2', rss_level=8),
 Character(_id='2f4', name='Dei 2F4', slot_number=4, ch=22, rss_order='V2', rss_level=8)]

In [5]:
from src.lib.rok_data import RssOrdersDB
RssOrdersDB().get_by_id("V1")

['gold', 'gold', 'stone', 'wood', 'food']

In [11]:
db.get_rss_order("V6")

[]